# Spark Exercises 02 — Lazy Execution & the Catalyst Plan

Polars taught you *lazy* evaluation with `scan` / `collect`. Spark takes the same idea much further: **every** DataFrame is lazy, and behind the scenes the **Catalyst optimizer** rewrites your plan before a single row is read. In this chapter you will *see* the plan with `explain()`, watch Catalyst merge filters and prune columns, and learn when to `cache()`.

**Data file:** `data/orders.csv`

---

> **Solutions version** — try it yourself first from `Exercises.ipynb`.

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.1


### 2. Read `data/orders.csv` (`header=True`, `inferSchema=True`) into `orders`.

In [2]:
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)

### 3. Build a **chain of transformations** without any action and store it in `pipeline`: keep only positive `quantity`, select a few columns, and add `revenue = quantity * unit_price`. Then evaluate `pipeline` — it returns instantly and shows only the schema. **No job ran yet.**

In [3]:
pipeline = (
    orders
    .filter(F.col("quantity") > 0)
    .select("order_id", "product_sku", "quantity", "unit_price")
    .withColumn("revenue", F.col("quantity") * F.col("unit_price"))
)
pipeline

DataFrame[order_id: string, product_sku: string, quantity: int, unit_price: double, revenue: double]

### 4. Now call an **action** — `pipeline.show(5)` — and the whole chain finally runs.

In [4]:
pipeline.show(5)

+----------+-----------+--------+----------+------------------+
|  order_id|product_sku|quantity|unit_price|           revenue|
+----------+-----------+--------+----------+------------------+
|ORD-000001|      BK302|      10|     32.58|325.79999999999995|
|ORD-000002|      ST500|      22|      1.43|31.459999999999997|
|ORD-000003|      PB024|       9|      1.14|             10.26|
|ORD-000004|      MG010|      21|      7.12|            149.52|
|ORD-000005|      PT044|      23|      8.17|            187.91|
+----------+-----------+--------+----------+------------------+
only showing top 5 rows



### 5. Print the **physical plan** with `pipeline.explain()`. Read it bottom-up: the bottom is the file scan, the top is the final step.

In [5]:
pipeline.explain()

== Physical Plan ==
*(1) Project [order_id#17, product_sku#20, quantity#21, unit_price#22, (cast(quantity#21 as double) * unit_price#22) AS revenue#42]
+- *(1) Filter (isnotnull(quantity#21) AND (quantity#21 > 0))
   +- FileScan csv [order_id#17,product_sku#20,quantity#21,unit_price#22] Batched: false, DataFilters: [isnotnull(quantity#21), (quantity#21 > 0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/tomerkadison/Desktop/hafifot_osint/spark_exercises/02_Lazy..., PartitionFilters: [], PushedFilters: [IsNotNull(quantity), GreaterThan(quantity,0)], ReadSchema: struct<order_id:string,product_sku:string,quantity:int,unit_price:double>




### 6. Print the **full set of plans** (parsed → analyzed → optimized → physical) with `pipeline.explain(mode="extended")`.

In [6]:
pipeline.explain(mode="extended")

== Parsed Logical Plan ==
'Project [order_id#17, product_sku#20, quantity#21, unit_price#22, ('quantity * 'unit_price) AS revenue#42]
+- Project [order_id#17, product_sku#20, quantity#21, unit_price#22]
   +- Filter (quantity#21 > 0)
      +- Relation [order_id#17,customer_id#18,order_ts#19,product_sku#20,quantity#21,unit_price#22,channel#23,payment_method#24,status#25,discount_code#26] csv

== Analyzed Logical Plan ==
order_id: string, product_sku: string, quantity: int, unit_price: double, revenue: double
Project [order_id#17, product_sku#20, quantity#21, unit_price#22, (cast(quantity#21 as double) * unit_price#22) AS revenue#42]
+- Project [order_id#17, product_sku#20, quantity#21, unit_price#22]
   +- Filter (quantity#21 > 0)
      +- Relation [order_id#17,customer_id#18,order_ts#19,product_sku#20,quantity#21,unit_price#22,channel#23,payment_method#24,status#25,discount_code#26] csv

== Optimized Logical Plan ==
Project [order_id#17, product_sku#20, quantity#21, unit_price#22, (cas

### 7. **Catalyst merges filters.** Build `two = orders.filter(quantity > 0).filter(unit_price > 5)` and call `two.explain()`. Notice the physical plan has **one** combined `Filter`, not two.

In [7]:
two = orders.filter(F.col("quantity") > 0).filter(F.col("unit_price") > 5)
two.explain()

== Physical Plan ==
*(1) Filter (((isnotnull(quantity#21) AND isnotnull(unit_price#22)) AND (quantity#21 > 0)) AND (unit_price#22 > 5.0))
+- FileScan csv [order_id#17,customer_id#18,order_ts#19,product_sku#20,quantity#21,unit_price#22,channel#23,payment_method#24,status#25,discount_code#26] Batched: false, DataFilters: [isnotnull(quantity#21), isnotnull(unit_price#22), (quantity#21 > 0), (unit_price#22 > 5.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/tomerkadison/Desktop/hafifot_osint/spark_exercises/02_Lazy..., PartitionFilters: [], PushedFilters: [IsNotNull(quantity), IsNotNull(unit_price), GreaterThan(quantity,0), GreaterThan(unit_price,5.0)], ReadSchema: struct<order_id:string,customer_id:string,order_ts:timestamp,product_sku:string,quantity:int,unit...




### 8. **Column pruning (projection pushdown).** Call `orders.select("order_id", "quantity").explain()` and look at the scan's `ReadSchema` — Spark only reads the two columns it needs.

In [8]:
orders.select("order_id", "quantity").explain()

== Physical Plan ==
FileScan csv [order_id#17,quantity#21] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/tomerkadison/Desktop/hafifot_osint/spark_exercises/02_Lazy..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,quantity:int>




### 9. How many **partitions** does `orders` have? (`orders.rdd.getNumPartitions()`) — this is how many chunks Spark processes in parallel.

In [9]:
orders.rdd.getNumPartitions()

1

### 10. Without caching, **every action recomputes the whole chain**. `cache()` the pipeline, then run an action (`count()`) to materialize it.

In [10]:
pipeline.cache()
pipeline.count()

7694

### 11. Check `pipeline.storageLevel` — now that it is cached you will see it is stored in memory.

In [11]:
pipeline.storageLevel

StorageLevel(True, True, False, True, 1)

### 12. Free the cache with `pipeline.unpersist()` and confirm `storageLevel` is back to none.

In [12]:
pipeline.unpersist()
pipeline.storageLevel

StorageLevel(False, False, False, False, 1)

### 13. You can also write expressions as SQL strings. Use `selectExpr` to compute `quantity * unit_price as revenue` for 5 rows.

In [13]:
orders.selectExpr("order_id", "quantity * unit_price as revenue").show(5)

+----------+------------------+
|  order_id|           revenue|
+----------+------------------+
|ORD-000001|325.79999999999995|
|ORD-000002|31.459999999999997|
|ORD-000003|             10.26|
|ORD-000004|            149.52|
|ORD-000005|            187.91|
+----------+------------------+
only showing top 5 rows

